In [1]:
# # 1. Installations and Dependencies

# # We need the OpenAI adapter because LM Studio mimics the OpenAI API structure
# %pip install -qU langchain-openai
# %pip install -qU openai

# # Core LangChain and LangGraph
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv

In [2]:
# 2. Environment setup and LLM initialization via LangChain
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# LM Studio Configuration
# The base_url must match the Local Server endpoint in LM Studio (usually port 1234)
# The api_key is required by the SDK but ignored by the local server (dummy string)
lm_studio_base = "http://localhost:1234/v1"
lm_studio_key = "lm-studio"

# Initialize the LLM using LangChain (ready for use with LangGraph + ReAct)
# We use ChatOpenAI because LM Studio mimics the OpenAI API architecture
llm = ChatOpenAI(
    model="google/gemma-3n-e4b",
    base_url=lm_studio_base,
    api_key=lm_studio_key,
    temperature=0
)

# Optional test (comment out if not needed)
print(f"Model target: google/gemma-3n-e4b at {lm_studio_base}")
print("Testing LLM connection:")
try:
    # Simple invocation to verify the local server is responsive
    print(llm.invoke("Hello, system check, just answer online.").content)
except Exception as e:
    print(f"Connection failed: {e}")
    print("Please ensure LM Studio server is running.")

Model target: google/gemma-3n-e4b at http://localhost:1234/v1
Testing LLM connection:
Online and ready! How can I help you? 😊 



In [3]:
# 3. Simplified LLM wrapper (Passthrough for Local Execution)
from langchain_core.messages import BaseMessage, HumanMessage
from typing import List, Union

class QuotaAwareLLM:
    """
    Simplified wrapper for LangChain LLMs.
    
    NOTE: Since we are using LM Studio (Local), the retry/quota logic 
    has been removed. This class is preserved to maintain compatibility 
    with downstream cells that expect this specific interface.
    """
    
    def __init__(self, llm):
        self.llm = llm

    def invoke_with_retry(
        self,
        messages: List[BaseMessage],
        **kwargs
    ) -> BaseMessage:
        """
        Direct pass-through to the local LLM.
        No retry logic needed for local execution.
        """
        # We simply invoke the model directly
        return self.llm.invoke(messages, **kwargs)

    def __call__(
        self,
        messages: Union[str, List[BaseMessage]],
        **kwargs
    ) -> str:
        """
        Simplified interface for agent nodes.
        Handles string-to-message conversion and returns string content.
        """
        if isinstance(messages, str):
            messages = [HumanMessage(content=messages)]
        
        response = self.llm.invoke(messages, **kwargs)
        return response.content if hasattr(response, "content") else str(response)

In [4]:
# 4. ReAct system prompt definition

REACT_SYSTEM_PROMPT = """
You are a reasoning agent following the ReAct (Reasoning + Acting) framework.

Workflow:
1. THINK: Analyze the question and plan your approach
2. ACT: Use available tools to gather information
3. OBSERVE: Process tool responses
4. Repeat until ready for final ANSWER

Available tools:
- query_stock_level(item): Check inventory stock for a product
- get_product_price(product): Get current price of a product

Response format:
Thought: [Your reasoning]
Action: tool_name[argument]
PAUSE

Observation: [Tool result will appear here]
Answer: [Final comprehensive answer]

Example:
User: What's the price of a gaming mouse and how many are in stock?
Thought: I need to get the price first, then check stock levels.
Action: get_product_price[gaming mouse]
PAUSE

Observation: PRICE: Gaming Mouse - $99.50 USD per unit
Thought: Now I need to check the inventory.
Action: query_stock_level[gaming mouse]
PAUSE

Observation: Stock available: 80 units of Gaming Mouse
Answer: The gaming mouse costs $99.50 each and we have 80 units in stock.
""".strip()

# Validate prompt integrity
assert "ReAct" in REACT_SYSTEM_PROMPT, "System prompt must contain ReAct framework description"
print(f"[INFO] ReAct system prompt loaded ({len(REACT_SYSTEM_PROMPT)} characters)")

[INFO] ReAct system prompt loaded (1033 characters)


In [5]:
# 5. Agent state definition for ReAct workflow
from typing import TypedDict, List, Optional, Dict, Any

class AgentState(TypedDict):
    """
    State structure for ReAct agent in LangGraph.
    Tracks the full reasoning cycle while remaining serializable.
    """
    input: str                     # Original user query
    messages: List[Dict[str, str]] # Conversation history [{"role": "...", "content": "..."}, ...]
    next_action: Optional[Dict[str, Any]]  # Parsed tool call: {"tool": "name", "args": {"arg": "value"}}
    observation: Optional[str]     # Tool response to be processed
    answer: Optional[str]          # Final answer when workflow completes
    
    # Kept for compatibility with downstream nodes, even though 
    # local execution doesn't require quota retries.
    attempts: int

In [6]:
# 6. Tool implementations with error handling and consistent formatting
import time
from typing import Dict, Any

def query_stock_level(item_name: str) -> str:
    """
    Simulates inventory lookup with consistent ReAct-compatible formatting.
    
    Args:
        item_name: Product name to check stock for (case-insensitive)
    
    Returns:
        Formatted string matching ReAct observation format
    """
    time.sleep(0.1)  # Simulate real API latency
    normalized = item_name.lower().strip().replace("-", " ").replace("_", " ")
    
    # Inventory database with display names
    stock_db: Dict[str, Dict[str, Any]] = {
        "monitor": {"display": "Monitor", "quantity": 75},
        "keyboard": {"display": "Keyboard", "quantity": 120},
        "mouse": {"display": "Mouse", "quantity": 80},
        "gaming mouse": {"display": "Gaming Mouse", "quantity": 80},
        "webcam": {"display": "Webcam", "quantity": 40},
        "headset": {"display": "Headset", "quantity": 60},
        "printer": {"display": "Printer", "quantity": 15},
        "laptop": {"display": "Laptop", "quantity": 25}
    }
    
    item = stock_db.get(normalized)
    if not item:
        return f"Observation: Item '{item_name}' not found in inventory system"
    
    return f"Stock available: {item['quantity']} units of {item['display']}"

def get_product_price(product: str) -> str:
    """
    Simulates price lookup with ReAct-compatible response formatting.
    
    Args:
        product: Product name for price lookup (case-insensitive)
    
    Returns:
        Formatted price string matching system prompt examples
    """
    time.sleep(0.1)  # Simulate real API latency
    normalized = product.lower().strip().replace("-", " ").replace("_", " ")
    
    # Price catalog with display names
    price_db: Dict[str, Dict[str, Any]] = {
        "monitor": {"display": "Monitor", "price": 999.90},
        "keyboard": {"display": "Keyboard", "price": 150.00},
        "mouse": {"display": "Mouse", "price": 45.00},
        "gaming mouse": {"display": "Gaming Mouse", "price": 99.50},
        "webcam": {"display": "Webcam", "price": 120.00},
        "headset": {"display": "Headset", "price": 180.00},
        "printer": {"display": "Printer", "price": 750.00}
    }
    
    item = price_db.get(normalized)
    if not item:
        return f"Observation: Product '{product}' not found in pricing catalog"
    
    return f"PRICE: {item['display']} - ${item['price']:.2f} USD per unit"

In [7]:
# 7. Tool validation tests with expected outputs
print("=== STOCK CHECK TESTS ===")
print(f"Keyboard stock: {query_stock_level('keyboard')}")
print(f"Monitor stock: {query_stock_level('monitor')}")
print(f"Invalid item test: {query_stock_level('smartphone')}\n")

print("=== PRICE CHECK TESTS ===")
print(f"Printer price: {get_product_price('printer')}")
print(f"Gaming mouse price: {get_product_price('gaming mouse')}")
print(f"Invalid product test: {get_product_price('tablet')}\n")

# Automated validation checks
assert "120 units of Keyboard" in query_stock_level("keyboard"), "Keyboard stock test failed"
assert "75 units of Monitor" in query_stock_level("monitor"), "Monitor stock test failed"
assert "not found" in query_stock_level("smartphone"), "Invalid item handling failed"

assert "$750.00 USD per unit" in get_product_price("printer"), "Printer price test failed"
assert "$99.50 USD per unit" in get_product_price("gaming mouse"), "Gaming mouse price test failed"
assert "not found" in get_product_price("tablet"), "Invalid product handling failed"

print("[SUCCESS] All tool validation tests passed!")

=== STOCK CHECK TESTS ===
Keyboard stock: Stock available: 120 units of Keyboard
Monitor stock: Stock available: 75 units of Monitor
Invalid item test: Observation: Item 'smartphone' not found in inventory system

=== PRICE CHECK TESTS ===
Printer price: PRICE: Printer - $750.00 USD per unit
Gaming mouse price: PRICE: Gaming Mouse - $99.50 USD per unit
Invalid product test: Observation: Product 'tablet' not found in pricing catalog

[SUCCESS] All tool validation tests passed!


In [18]:
# 8. Optimized ReAct agent (Regex Patch for Robust Tool Calling)
import re
from typing import Dict, Callable, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

class ReActAgent:
    def __init__(
        self,
        robust_llm: Callable,
        system_prompt: str,
        tools: Dict[str, Callable]
    ):
        self.robust_llm = robust_llm
        self.system_prompt = system_prompt
        self.tools = tools
        self.messages = [] 
        self.product_aliases = {
            "keyboards": "keyboard", "mice": "mouse", "monitors": "monitor",
            "headsets": "headset", "printers": "printer", "laptops": "laptop", "webcams": "webcam"
        }
    
    def normalize_product_name(self, name: str) -> str:
        cleaned = name.lower().strip().replace("-", " ").replace("_", " ")
        return self.product_aliases.get(cleaned, cleaned)
    
    def extract_action(self, response: str) -> str:
        """
        Robust Regex:
        1. Allows [brackets] OR (parentheses) - Models confuse these often.
        2. Allows empty arguments (.*?) instead of requiring content (.+?).
        """
        # Pattern explanation:
        # Action:          Match literal "Action:"
        # \s*              Optional spaces
        # ([a-zA-Z0-9_]+)  Capture Group 1: Tool Name
        # \s*              Optional spaces
        # (?:\[|\()        Match OPEN bracket OR parenthesis
        # \s*              Optional spaces
        # (.*?)            Capture Group 2: Arguments (Empty allowed)
        # \s*              Optional spaces
        # (?:\]|\))        Match CLOSE bracket OR parenthesis
        pattern = r"Action:\s*([a-zA-Z0-9_]+)\s*(?:\[|\()\s*(.*?)\s*(?:\]|\))"
        
        if match := re.search(pattern, response, re.IGNORECASE):
            tool = match.group(1).strip()
            args = match.group(2).strip()
            # Reconstruct to standard internal format for processing
            return f"{tool}[{args}]"
        return ""
    
    def _get_langchain_messages(self) -> List:
        lc_messages = []
        first_user_msg_index = -1
        
        # Find first user message to inject system prompt
        for i, m in enumerate(self.messages):
            if m["role"] == "user":
                first_user_msg_index = i
                break
        
        for i, m in enumerate(self.messages):
            content = m["content"]
            if i == first_user_msg_index:
                content = f"{self.system_prompt}\n\nUSER QUERY: {content}"
            
            if m["role"] == "user":
                lc_messages.append(HumanMessage(content=content))
            elif m["role"] == "assistant":
                lc_messages.append(AIMessage(content=content))
            elif m["role"] == "observation":
                lc_messages.append(HumanMessage(content=f"Observation: {content}"))
                
        return lc_messages

    def run(self, query: str, max_iterations: int = 5) -> str:
        print(f"\n🔍 Query: {query}")
        self.messages = [{"role": "user", "content": query}] 
        
        for iteration in range(max_iterations):
            print(f"\n🔄 Iteration {iteration + 1}/{max_iterations}")
            
            lc_messages = self._get_langchain_messages()
            
            try:
                response_msg = self.robust_llm.llm.invoke(
                    lc_messages, 
                    stop=["Observation:", "PAUSE", "Observation"]
                )
                response = response_msg.content
            except Exception as e:
                return f"AGENT ERROR: {str(e)}"
            
            self.messages.append({"role": "assistant", "content": response})
            print(f"🤖 Response: {response[:200]}...")
            
            if "Answer:" in response:
                return response.split("Answer:", 1)[1].strip()
            
            if action_text := self.extract_action(response):
                print(f"⚡ Action: {action_text}")
                
                if match := re.match(r"(\w+)\s*\[\s*(.*?)\s*\]", action_text):
                    tool_name = match.group(1)
                    raw_arg = match.group(2)
                    
                    # Normalize only if arg exists
                    tool_arg = self.normalize_product_name(raw_arg) if raw_arg else ""
                    
                    if tool_name not in self.tools:
                        observation = f"Tool '{tool_name}' not found. Available: {list(self.tools.keys())}"
                    else:
                        try:
                            # Handle tools with/without args
                            if tool_arg:
                                observation = self.tools[tool_name](tool_arg)
                            else:
                                # Call without args if empty
                                try:
                                    observation = self.tools[tool_name]()
                                except TypeError:
                                    # Fallback: tool might expect an arg even if empty
                                    observation = self.tools[tool_name]("")
                                    
                            print(f"✅ Tool executed: {tool_name}({tool_arg})")
                        except Exception as e:
                            observation = f"TOOL ERROR: {str(e)}"
                else:
                    observation = "Invalid action format."
                
                print(f"🔍 Obs: {observation}")
                self.messages.append({"role": "observation", "content": observation})
                continue
            
            if iteration < max_iterations - 1:
                self.messages.append({"role": "observation", "content": "Please continue. Use Action: tool[arg] or Answer: ..."})
        
        return f"MAX ITERATIONS. Last: {response[:100]}"

# Re-init
robust_llm = QuotaAwareLLM(llm=llm)
# Note: We re-initialize immediately to apply the patch
react_agent = ReActAgent(
    robust_llm=robust_llm, 
    # We use a placeholder here; the actual prompt comes from Cell 15 later
    system_prompt="Placeholder", 
    tools={"query_stock_level": query_stock_level, "get_product_price": get_product_price}
)
print("[✅] Patched ReAct Agent (Regex v2) Ready")

[✅] Patched ReAct Agent (Regex v2) Ready


In [9]:
# 9. Execution wrapper with safe defaults
def run_react_agent(query: str, max_iterations: int = 5) -> str:
    """Safe execution interface for notebook testing"""
    try:
        # Calls the updated ReAct agent
        return react_agent.run(query, max_iterations=max_iterations)
    except Exception as e:
        return f"EXECUTION FAILED: {str(e)}"

In [10]:
# 10. Practical test suite (Optimized for Local Execution)
import time

print("="*50)
print("🧪 NOTEBOOK-OPTIMIZED REACT AGENT TESTS (LM STUDIO)")
print("="*50)

test_cases = [
    ("Stock check (singular)", "How many keyboard are in stock?"),
    ("Price query", "What is the price of a headset?"),
    ("Plural handling", "Do we have monitors in stock?"),
    ("Error case", "What's the price of a chair?")
]

for title, query in test_cases:
    print(f"\n{'='*200}")
    print(f"🔹 TEST: {title}")
    print(f"❓ Query: {query}")
    
    # Increased iterations to 5 to give the local model breathing room for reasoning
    result = run_react_agent(query, max_iterations=5)
    print(f"\n✅ RESULT: {result}")
    
    # Note: Sleep removed. Local models do not have rate limits/quotas.

print("\n" + "="*200)
print("[🎉] TEST SUITE COMPLETED")
print("="*200)

🧪 NOTEBOOK-OPTIMIZED REACT AGENT TESTS (LM STUDIO)

🔹 TEST: Stock check (singular)
❓ Query: How many keyboard are in stock?

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/5
🤖 Response: Thought: I need to find the stock level for keyboards.
Action: query_stock_level[keyboard]
...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Obs: Stock available: 120 units of Keyboard

🔄 Iteration 2/5
🤖 Response: Thought: I have the stock level for keyboards. Now I can provide the answer.
Answer: There are 120 keyboards in stock....

✅ RESULT: There are 120 keyboards in stock.

🔹 TEST: Price query
❓ Query: What is the price of a headset?

🔍 Query: What is the price of a headset?

🔄 Iteration 1/5
🤖 Response: Thought: I need to find the price of a headset. I will use the get_product_price tool for this.
Action: get_product_price[headset]
...
⚡ Action: get_product_price[headset]
✅ Tool executed: get_product_price(headset)
🔍 Obs: PRICE: Headset - $180.00 USD 

In [11]:
# 11. Agent execution with history tracking (Compatible with Local Strategy)
from typing import Tuple, List, Dict

# Global tools definition (preserved for compatibility)
available_tools = {
    "query_stock_level": query_stock_level,
    "get_product_price": get_product_price
}

def run_react_agent_with_history(
    question: str,
    max_iterations: int = 5  # Increased to 5 for Local Models
) -> Tuple[str, List[Dict[str, str]]]:
    """
    Execute agent with state reset per call and return answer + clean history.
    """
    global react_agent
    
    # Note: We removed the manual 'system' prompt insertion here.
    # The new ReActAgent.run() method handles state initialization 
    # and prompt injection internally to satisfy the local model.
    
    # Execute with controlled iterations
    final_answer = react_agent.run(question, max_iterations=max_iterations)
    
    # Clean history: We filter out observations to show a cleaner chat log,
    # but keep the reasoning (Assistant) and the query (User).
    clean_history = [
        msg for msg in react_agent.messages
        if msg["role"] not in ["system", "observation"]
    ]
    
    return final_answer, clean_history

def print_full_history(history: List[Dict[str, str]]):
    """Print history with notebook-friendly formatting"""
    print("\n" + "="*60)
    print("📜 CONVERSATION HISTORY")
    print("="*60)
    
    for i, entry in enumerate(history):
        role = entry["role"].upper()
        content = entry["content"].strip()
        print(f"\n[{i+1}] {role}:")
        # Truncate very long contents for display clarity
        display_content = content[:500] + "..." if len(content) > 500 else content
        print(display_content)
        print("-"*40)
    
    print("\n" + "="*60)

print("[✅] History tracking functions ready (Local Model Adapted)")

[✅] History tracking functions ready (Local Model Adapted)


In [12]:
# 12. Example usage with clean output formatting
print("="*70)
print("🔍 DEMONSTRATION: Agent execution with history tracking (LM Studio)")
print("="*70)

example_question = "How many keyboard are in stock?"
print(f"\n❓ USER QUESTION: {example_question}")

# Execute agent with history tracking
# Increased max_iterations to 5 to ensure local models have room to reason
final_answer, full_history = run_react_agent_with_history(
    example_question,
    max_iterations=5  
)

# Print final answer prominently
print(f"\n{'='*50}")
print("✅ FINAL AGENT ANSWER")
print(f"{'='*50}")
print(f"➤ {final_answer}")
print(f"{'='*50}")

# Print clean interaction history
print_full_history(full_history)

print("\n💡 NOTE: History shows only user/assistant interactions (internal tool calls are filtered)")

🔍 DEMONSTRATION: Agent execution with history tracking (LM Studio)

❓ USER QUESTION: How many keyboard are in stock?

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/5
🤖 Response: Thought: I need to find the stock level for keyboards.
Action: query_stock_level[keyboard]
...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Obs: Stock available: 120 units of Keyboard

🔄 Iteration 2/5
🤖 Response: Thought: I have the stock level for keyboards. Now I can provide the answer.
Answer: There are 120 keyboards in stock....

✅ FINAL AGENT ANSWER
➤ There are 120 keyboards in stock.

📜 CONVERSATION HISTORY

[1] USER:
How many keyboard are in stock?
----------------------------------------

[2] ASSISTANT:
Thought: I need to find the stock level for keyboards.
Action: query_stock_level[keyboard]
----------------------------------------

[3] ASSISTANT:
Thought: I have the stock level for keyboards. Now I can provide the answer.
Answer: There are 120 keyboard

In [13]:
# 13. Production-ready function to find most expensive product
def find_most_expensive_product() -> str:
    """
    Returns the name and price of the most expensive product in inventory.
    Uses existing price lookup tool to avoid data duplication.
    """
    # Products to check (matches our tool's known items)
    products_to_check = [
        "monitor", "keyboard", "gaming mouse", 
        "webcam", "headset", "printer"
    ]
    
    product_prices = {}
    
    # Get prices using our existing tool (avoids data duplication)
    for product in products_to_check:
        price_response = get_product_price(product)
        
        # Parse price from tool's response format: "PRICE: Product - $123.45 USD per unit"
        if "PRICE:" in price_response and "$" in price_response:
            try:
                # Extract price value (e.g., "$999.90" -> 999.90)
                price_str = price_response.split("$")[1].split(" ")[0]
                price_value = float(price_str.replace(",", ""))
                product_prices[product] = price_value
            except (IndexError, ValueError) as e:
                print(f"[WARNING] Failed to parse price for '{product}': {price_response}")
                continue
    
    # Safety checks
    if not product_prices:
        return "ERROR: Could not retrieve any product prices from inventory system."
    
    if len(product_prices) < len(products_to_check):
        print(f"[INFO] Retrieved prices for {len(product_prices)}/{len(products_to_check)} products")
    
    # Find most expensive product
    most_expensive = max(product_prices.items(), key=lambda x: x[1])
    product_name, price_value = most_expensive
    
    # Format with proper capitalization (handles multi-word products)
    display_name = " ".join(word.capitalize() for word in product_name.split())
    
    return (
        f"The most expensive product is '{display_name}' "
        f"with a price of ${price_value:,.2f} USD."
    )

# Quick validation test
print("[✅] Function defined. Test result:")
print(find_most_expensive_product())

[✅] Function defined. Test result:
The most expensive product is 'Monitor' with a price of $999.90 USD.


In [14]:
# 14. Course-compatible ReAct execution wrapper (LM Studio Adapted)
import re

def _normalize_product_name(name: str) -> str:
    """Normalize product names for tool execution (matches our ReActAgent logic)"""
    cleaned = name.lower().strip().replace("-", " ").replace("_", " ")
    aliases = {
        "keyboards": "keyboard",
        "mice": "mouse",
        "monitors": "monitor",
        "headsets": "headset",
        "printers": "printer",
        "laptops": "laptop",
        "webcams": "webcam"
    }
    return aliases.get(cleaned, cleaned)

def run_react_agent(question: str, max_iterations: int = 5, reset_history: bool = True) -> str:
    """
    Course-compatible wrapper tailored for Local LLM execution.
    
    Key adaptations:
    - Delegates history management to react_agent.run() (required for local prompt injection)
    - Defaults to higher iteration count (5) for local model reasoning
    - Maintains response post-processing
    """
    global react_agent
    
    # NOTE: In our Local Model strategy (Cell 8), the .run() method automatically
    # handles history reset and prompt injection. We do NOT manually set
    # react_agent.messages here to avoid conflicting with the "User-Prompt" strategy.
    
    # Execute
    final_answer = react_agent.run(question, max_iterations=max_iterations)
    
    # Post-process answer to match course expectations:
    # 1. Remove prefixes like "Answer: " if present
    if final_answer.startswith("Answer:"):
        final_answer = final_answer.split("Answer:", 1)[1].strip()
    
    # 2. Handle tool errors gracefully
    error_patterns = [
        "not found in inventory",
        "not found in pricing catalog",
        "ERROR:",
        "PARSING ERROR:",
        "TOOL ERROR:"
    ]
    
    if any(pattern in final_answer for pattern in error_patterns):
        # Convert tool errors to user-friendly messages
        lower_ans = final_answer.lower()
        if "keyboard" in lower_ans and "not found" in lower_ans:
            return "I cannot find 'keyboard' in the inventory. Please verify the product name."
        elif "not found" in lower_ans:
            product_match = re.search(r"'(.*?)'", final_answer)
            product = product_match.group(1) if product_match else "the product"
            return f"I cannot find '{product}' in the inventory. Please verify the product name."
    
    return final_answer

print("[✅] Course-compatible wrapper adapted for Local Model strategy")

[✅] Course-compatible wrapper adapted for Local Model strategy


In [19]:
# 15. CORRECTED: Tool registration + comprehensive test suite (LM Studio Optimized)
import time

print("="*70)
print("🔧 AGENT TOOL REGISTRATION")
print("="*70)

# 1. Register new tool with the agent
print("\n[+] Registering new tool: find_most_expensive_product")
react_agent.tools["find_most_expensive_product"] = find_most_expensive_product

# 2. Update system prompt to include new tool
# We stick to the prompt format that worked with Gemma, simply adding the new tool.
updated_prompt = """
You are a reasoning agent following the ReAct (Reasoning + Acting) framework.

Workflow:
1. THINK: Analyze the question and plan your approach
2. ACT: Use available tools to gather information using the syntax: Action: tool_name[argument]
3. OBSERVE: Process tool responses
4. Repeat until ready for final ANSWER

Available tools:
- query_stock_level(item): Check inventory stock for a product
- get_product_price(product): Get current price of a product
- find_most_expensive_product(): Returns the most expensive product in inventory (no arguments needed)

Response format:
Thought: [Your reasoning]
Action: tool_name[argument]
PAUSE

Observation: [Tool result will appear here]
Answer: [Final comprehensive answer]

Example:
User: What's the price of a gaming mouse?
Thought: I need to get the price.
Action: get_product_price[gaming mouse]
PAUSE

Observation: PRICE: Gaming Mouse - $99.50 USD per unit
Answer: The gaming mouse costs $99.50.
""".strip()

# Update the prompt in the agent config
react_agent.system_prompt = updated_prompt
# NOTE: We do NOT manually set react_agent.messages here. 
# The dynamic injection strategy in Cell 8 handles this automatically.

print("[✅] Tool registration completed successfully")
print(f"Available tools: {list(react_agent.tools.keys())}")
print(f"System prompt updated with new tool description")

# 3. Run comprehensive tests
print("\n" + "="*300)
print("🧪 REACT AGENT INTEGRATION TESTS (LOCAL)")
print("="*300)

test_cases = [
    ("Stock Check", "How many keyboard are in stock?"),
    ("Price Query", "What is the price of a headset?"),
    ("Error Handling", "Do we have chairs in stock?"),
    ("Complex Query", "What is the most expensive product?")
]

for test_name, question in test_cases:
    print(f"\n{'='*300}")
    print(f"🔹 TEST: {test_name}")
    print(f"❓ Query: {question}")
    print("-"*300)
    
    try:
        # Use notebook-optimized parameters for local execution
        answer = run_react_agent(
            question,
            max_iterations=5, # Increased for complex queries
            reset_history=True
        )
        print(f"\n✅ RESULT: {answer}")
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        answer = f"EXECUTION FAILED: {str(e)}"
    
    # Sleep removed: Local models don't have rate limits!

print(f"\n{'='*300}")
print("[🎉] ALL TESTS COMPLETED SUCCESSFULLY")
print(f"{'='*300}")

🔧 AGENT TOOL REGISTRATION

[+] Registering new tool: find_most_expensive_product
[✅] Tool registration completed successfully
Available tools: ['query_stock_level', 'get_product_price', 'find_most_expensive_product']
System prompt updated with new tool description

🧪 REACT AGENT INTEGRATION TESTS (LOCAL)

🔹 TEST: Stock Check
❓ Query: How many keyboard are in stock?
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

🔍 Query: How many keyboard are in stock?

🔄 Iteration 1/5
🤖 Response: Thought: I need to find the stock level of keyboards.
Action: query_stock_level[keyboard]
...
⚡ Action: query_stock_level[keyboard]
✅ Tool executed: query_stock_level(keyboard)
🔍 Obs: Stock available: 120 units of Keyboard

🔄 Iteration 2/5
🤖 Response: Thought:

In [20]:
# 16. Interactive agent conversation loop (LM Studio Optimized)
import time
from IPython.display import clear_output

def iniciar_conversacao_com_agente():
    """
    Interactive loop optimized for Local LLM execution.
    - Removes artificial delays (no rate limits on localhost)
    - increased iteration count for complex reasoning
    - Robust error handling for local inference issues
    """
    print("="*80)
    print("💬 INTERACTIVE INVENTORY AGENT (LM STUDIO LOCAL)")
    print("="*80)
    print("\nAvailable commands:")
    print("- Ask any inventory question (stock levels, prices, most expensive item)")
    print("- Type 'reset' to clear conversation context")
    print("- Type 'tools' to see available functions")
    print("- Type 'exit' or 'sair' to end conversation")
    print("\n" + "*" * 80)
    
    conversation_history = []
    
    while True:
        # Get user input
        try:
            pergunta_usuario = input("\n[You]: ").strip()
        except KeyboardInterrupt:
            print("\n\n⚠️ Conversation interrupted. Type 'exit' to close properly.")
            continue
        
        # Exit condition
        if pergunta_usuario.lower() in ['exit', 'sair', 'quit', 'parar']:
            print("\n" + "="*50)
            print("👋 AGENT SESSION ENDED")
            print("="*50)
            break
        
        # Special commands
        if pergunta_usuario.lower() == 'reset':
            # Note: Our local agent logic resets automatically per run, 
            # but this gives feedback to the user.
            conversation_history = []
            print("\n[Agent]: ✅ Context reset. Ready for new topic.")
            continue
        
        if pergunta_usuario.lower() == 'tools':
            tools_list = ", ".join(react_agent.tools.keys())
            print(f"\n[Agent]: 🔧 Available tools: {tools_list}")
            continue
        
        # No artificial waiting time needed for Local LLM!
        print("\n[Agent]: 🔄 Thinking locally...")
        
        try:
            # Execute agent with optimized parameters for local model
            resposta_agente = run_react_agent(
                pergunta_usuario,
                max_iterations=5,  # Giving the model space to think
                reset_history=True # Ensure fresh prompt injection for stability
            )
            
            # Record interaction
            conversation_history.append({
                "user": pergunta_usuario,
                "agent": resposta_agente
            })
            
            # Display response with clear formatting
            print("\n" + "-"*80)
            print(f"[Agent]: {resposta_agente}")
            print("-"*80)
            
        except Exception as e:
            error_msg = str(e)
            print(f"\n[Agent]: ❌ ERROR: {error_msg}")
            print("Tip: If the model hallucinates, try simplifying the question.")
        
        print("\n💡 Tip: Type 'exit' to end, 'reset' to clear, or 'tools' to list functions")

# Start interactive session
if __name__ == "__main__":
    print("[✅] Interactive agent module loaded.")
    print("👉 Running interaction loop now...")
    iniciar_conversacao_com_agente()

[✅] Interactive agent module loaded.
👉 Running interaction loop now...
💬 INTERACTIVE INVENTORY AGENT (LM STUDIO LOCAL)

Available commands:
- Ask any inventory question (stock levels, prices, most expensive item)
- Type 'reset' to clear conversation context
- Type 'tools' to see available functions
- Type 'exit' or 'sair' to end conversation

********************************************************************************



[You]:  can you make a list of product in stock?



[Agent]: 🔄 Thinking locally...

🔍 Query: can you make a list of product in stock?

🔄 Iteration 1/5
🤖 Response: Thought: The user wants a list of products currently in stock. I should use the `query_stock_level` tool to get this information.
Action: query_stock_level[]
...
⚡ Action: query_stock_level[]
✅ Tool executed: query_stock_level()
🔍 Obs: Observation: Item '' not found in inventory system

🔄 Iteration 2/5
🤖 Response: Thought: The previous query returned an error, indicating that the `query_stock_level` tool needs a specific item name as an argument. I need to provide a valid item name. Since the user asked for a l...
⚡ Action: find_most_expensive_product[]
✅ Tool executed: find_most_expensive_product()
🔍 Obs: The most expensive product is 'Monitor' with a price of $999.90 USD.

🔄 Iteration 3/5
🤖 Response: Thought: Now that I know the most expensive product is "Monitor", I can use `query_stock_level` to check if it's in stock. Then, I will present the list of products in stock.
A


[You]:  ok i want to print a file, do you have any printer?



[Agent]: 🔄 Thinking locally...

🔍 Query: ok i want to print a file, do you have any printer?

🔄 Iteration 1/5
🤖 Response: Thought: The user wants to print a file and is asking if there are any printers available. I need to check the inventory for printers.
Action: query_stock_level[printer]
...
⚡ Action: query_stock_level[printer]
✅ Tool executed: query_stock_level(printer)
🔍 Obs: Stock available: 15 units of Printer

🔄 Iteration 2/5
🤖 Response: Thought: I know there are 15 printers in stock. Therefore, I can confirm that printers are available.
Answer: Yes, we have 15 printers in stock.
...

--------------------------------------------------------------------------------
[Agent]: Yes, we have 15 printers in stock.
--------------------------------------------------------------------------------

💡 Tip: Type 'exit' to end, 'reset' to clear, or 'tools' to list functions


...


[Agent]: 🔄 Thinking locally...

🔍 Query: what is the sum of 1 monitor and 1 printer?

🔄 Iteration 1/5
🤖 Response: Thought: The question asks for the sum of the prices of a monitor and a printer. I need to find the price of each item first using the available tools, then add them together.
Action: get_product_pric...
⚡ Action: get_product_price[monitor]
✅ Tool executed: get_product_price(monitor)
🔍 Obs: PRICE: Monitor - $999.90 USD per unit

🔄 Iteration 2/5
🤖 Response: Thought: Now I need to find the price of a printer.
Action: get_product_price[printer]
...
⚡ Action: get_product_price[printer]
✅ Tool executed: get_product_price(printer)
🔍 Obs: PRICE: Printer - $750.00 USD per unit

🔄 Iteration 3/5
🤖 Response: Thought: I have the prices of both items now. I can calculate their sum.
Action: None
...

🔄 Iteration 4/5
🤖 Response: Thought: I have the prices of a monitor and a printer. I need to calculate their sum and provide the answer.
Action: None
ANSWER: The sum of a monitor and a prin


[You]:  do you have any gamer acessory?



[Agent]: 🔄 Thinking locally...

🔍 Query: do you have any gamer acessory?

🔄 Iteration 1/5
🤖 Response: Thought: The user is asking if we have any gamer accessories. I should check the inventory for all items and see if there are any that are categorized as gamer accessories. Since I don't have a specif...
⚡ Action: find_most_expensive_product[]
✅ Tool executed: find_most_expensive_product()
🔍 Obs: The most expensive product is 'Monitor' with a price of $999.90 USD.

🔄 Iteration 2/5
🤖 Response: Thought: Based on the observation, the most expensive product is a monitor. While monitors can be used as gamer accessories, it's not definitively a gamer accessory. I should clarify with the user if ...

--------------------------------------------------------------------------------
[Agent]: The most expensive product in our inventory is a Monitor, priced at $999.90 USD.
--------------------------------------------------------------------------------

💡 Tip: Type 'exit' to end, 'reset' to clear


[You]:  exit



👋 AGENT SESSION ENDED
